# MS4 "SILENCE" Algorithm Analysis (CODE:023A)

Default A-RAM algorithm loaded at boot by `sam_aram_load_alg0`.
Program header at CODE:022E names it "SILENCE".

In [8]:
import sys
sys.path.insert(0, '..')
from sam8905_aram_decoder import decode_instruction, decode_algorithm, format_instruction

# Raw bytes from CODE:023A - direct from Ghidra memory_read hexBytes
# (spaces removed, verified 64 bytes = 128 hex chars)
ghidra_hex = "6F 08 BF 00 DF 0A 3F 11 DF 12 BF 18 DF 13 FB 7E 7D 68 F7 70 BD 78 EF 7C D7 7B 77 48 FF 7F BF 7C F7 70 EF 50 F7 7C 7F 7A BF 7C 7F 7A AF 58 DF 62 3F 40 7F 7A B7 38 DF 3B FF 7F FE 7E FF 7F FF 7F"
raw_bytes = bytes.fromhex(ghidra_hex.replace(' ', ''))
assert len(raw_bytes) == 64, f"Expected 64 bytes, got {len(raw_bytes)}"

# Reconstruct 15-bit A-RAM words (stored as [lo, hi] pairs)
words = []
for i in range(0, 64, 2):
    word = raw_bytes[i] | (raw_bytes[i+1] << 8)
    words.append(word)

# Decode all instructions
instructions = [decode_instruction(w) for w in words]

print(decode_algorithm(words, 0))

=== Algorithm 0 ===

PC00: 086F  RM 1, <WA, WPHI>
PC01: 00BF  RM 0, <WB>
PC02: 0ADF  RADD 1, <WM>
PC03: 113F  RM 2, <WA, WB, WSP> ***
PC04: 12DF  RADD 2, <WM>
PC05: 18BF  RM 3, <WB>
PC06: 13DF  RADD 2, <WM, WSP> ***
PC07: 7EFB  RSP, <clearB>
PC08: 687D  RM 13, <WA, WWF>
PC09: 70F7  RM 14, <WXY>
PC10: 78BD  RM 15, <WB, WWF>
PC11: 7CEF  RP, <WPHI>
PC12: 7BD7  RADD 15, <WM, WXY, WSP> ***
PC13: 4877  RM 9, <WA, WXY>
PC14: 7FFF  RSP, <WSP> ***
PC15: 7CBF  RP, <WB>
PC16: 70F7  RM 14, <WXY>
PC17: 50EF  RM 10, <WPHI>
PC18: 7CF7  RP, <WXY>
PC19: 7A7F  RADD, <WA>
PC20: 7CBF  RP, <WB>
PC21: 7A7F  RADD, <WA>
PC22: 58AF  RM 11, <WB, WPHI>
PC23: 62DF  RADD 12, <WM>
PC24: 403F  RM 8, <WA, WB>
PC25: 7A7F  RADD, <WA>
PC26: 38B7  RM 7, <WB, WXY>
PC27: 3BDF  RADD 7, <WM, WSP> ***
PC28: 7FFF  RSP, <WSP> ***
PC29: 7EFE  RSP, <WACC>
PC30: 7FFF  RSP, <WSP> ***
PC31: 7FFF  RSP, <WSP> ***


In [9]:
# Structured analysis table
print(f"{'PC':>3} {'Word':>6} {'Emit':>4} {'MAD':>3} {'WA':>2} {'WB':>2} {'WM':>2} {'Φ':>2} {'XY':>2} {'SP':>2} {'cB':>2} {'WF':>2} {'AC':>2}")
print("-" * 55)

for pc, (w, inst) in enumerate(zip(words, instructions)):
    emit_short = {'RM': 'RM', 'RADD': 'RA', 'RP': 'RP', 'RSP': 'RS'}[inst.emitter]
    mad_str = f"{inst.mad:2d}" if inst.emitter in ('RM', 'RADD') else " -"
    
    def dot(v): return '●' if v else '·'
    
    print(f"{pc:3d} 0x{w:04X} {emit_short:>4} {mad_str} "
          f"{dot(inst.wa):>2} {dot(inst.wb):>2} {dot(inst.wm):>2} "
          f"{dot(inst.wphi):>2} {dot(inst.wxy):>2} {dot(inst.wsp):>2} "
          f"{dot(inst.clear_b):>2} {dot(inst.wwf):>2} {dot(inst.wacc):>2}")

 PC   Word Emit MAD WA WB WM  Φ XY SP cB WF AC
-------------------------------------------------------
  0 0x086F   RM  1  ●  ·  ·  ●  ·  ·  ·  ·  ·
  1 0x00BF   RM  0  ·  ●  ·  ·  ·  ·  ·  ·  ·
  2 0x0ADF   RA  1  ·  ·  ●  ·  ·  ·  ·  ·  ·
  3 0x113F   RM  2  ●  ●  ·  ·  ·  ●  ·  ·  ·
  4 0x12DF   RA  2  ·  ·  ●  ·  ·  ·  ·  ·  ·
  5 0x18BF   RM  3  ·  ●  ·  ·  ·  ·  ·  ·  ·
  6 0x13DF   RA  2  ·  ·  ●  ·  ·  ●  ·  ·  ·
  7 0x7EFB   RS  -  ·  ·  ·  ·  ·  ·  ●  ·  ·
  8 0x687D   RM 13  ●  ·  ·  ·  ·  ·  ·  ●  ·
  9 0x70F7   RM 14  ·  ·  ·  ·  ●  ·  ·  ·  ·
 10 0x78BD   RM 15  ·  ●  ·  ·  ·  ·  ·  ●  ·
 11 0x7CEF   RP  -  ·  ·  ·  ●  ·  ·  ·  ·  ·
 12 0x7BD7   RA 15  ·  ·  ●  ·  ●  ●  ·  ·  ·
 13 0x4877   RM  9  ●  ·  ·  ·  ●  ·  ·  ·  ·
 14 0x7FFF   RS  -  ·  ·  ·  ·  ·  ●  ·  ·  ·
 15 0x7CBF   RP  -  ·  ●  ·  ·  ·  ·  ·  ·  ·
 16 0x70F7   RM 14  ·  ·  ·  ·  ●  ·  ·  ·  ·
 17 0x50EF   RM 10  ·  ·  ·  ●  ·  ·  ·  ·  ·
 18 0x7CF7   RP  -  ·  ·  ·  ·  ●  ·  ·  ·  ·
 19 0x7A7F   RA 15  ●  

In [10]:
# D-RAM slot usage analysis
print("=== D-RAM Slot Access Pattern ===\n")

dram_reads = {}  # mad -> list of PCs
dram_writes = {}  # mad -> list of PCs

for pc, inst in enumerate(instructions):
    if inst.emitter == 'RM':
        dram_reads.setdefault(inst.mad, []).append(pc)
    elif inst.emitter == 'RADD':
        # RADD reads MAD, adds to A, writes back to MAD
        dram_reads.setdefault(inst.mad, []).append(pc)
        dram_writes.setdefault(inst.mad, []).append(pc)

print(f"{'Slot':>4} {'Read at PCs':<20} {'Write at PCs':<20}")
print("-" * 50)
all_slots = sorted(set(list(dram_reads.keys()) + list(dram_writes.keys())))
for slot in all_slots:
    r = ', '.join(str(x) for x in dram_reads.get(slot, []))
    w = ', '.join(str(x) for x in dram_writes.get(slot, []))
    print(f"{slot:4d} {r:<20} {w:<20}")

print(f"\n=== Receiver Statistics ===\n")
stats = {
    'WA (load A)': sum(1 for i in instructions if i.wa),
    'WB (load B)': sum(1 for i in instructions if i.wb),
    'WM (write M/DRAM)': sum(1 for i in instructions if i.wm),
    'WPHI (phase accum)': sum(1 for i in instructions if i.wphi),
    'WXY (mix atten)': sum(1 for i in instructions if i.wxy),
    'WSP (special)': sum(1 for i in instructions if i.wsp),
    'clearB': sum(1 for i in instructions if i.clear_b),
    'WWF (waveform)': sum(1 for i in instructions if i.wwf),
    'WACC (output accum)': sum(1 for i in instructions if i.wacc),
}
for name, count in stats.items():
    print(f"  {name:<22}: {count:2d}")

print(f"\n=== Emitter Distribution ===\n")
emit_counts = {}
for inst in instructions:
    emit_counts[inst.emitter] = emit_counts.get(inst.emitter, 0) + 1
for e, c in sorted(emit_counts.items()):
    print(f"  {e:<6}: {c:2d}")

=== D-RAM Slot Access Pattern ===

Slot Read at PCs          Write at PCs        
--------------------------------------------------
   0 1                                        
   1 0, 2                 2                   
   2 3, 4, 6              4, 6                
   3 5                                        
   7 26, 27               27                  
   8 24                                       
   9 13                                       
  10 17                                       
  11 22                                       
  12 23                   23                  
  13 8                                        
  14 9, 16                                    
  15 10, 12, 19, 21, 25   12, 19, 21, 25      

=== Receiver Statistics ===

  WA (load A)           :  8
  WB (load B)           :  9
  WM (write M/DRAM)     :  6
  WPHI (phase accum)    :  4
  WXY (mix atten)       :  6
  WSP (special)         :  8
  clearB                :  1
  WWF (waveform)       

In [15]:
# Signal flow interpretation - corrected decode
print("=== SILENCE Algorithm Signal Flow ===\n")

print("Phase 1: Pitch / Phase Accumulation (PC 0-6)")
print("-" * 55)
print("  PC00: A = DRAM[1];  phase += A")
print("  PC01: B = DRAM[0]")
print("  PC02: DRAM[1] = A + B            ; phase += pitch_inc")
print("  PC03: A = B = DRAM[2]; WSP")
print("  PC04: DRAM[2] = A + B            ; 2nd phase accum")
print("  PC05: B = DRAM[3]")
print("  PC06: DRAM[2] = A + B; WSP       ; phase + mod_depth")
print()
print("  DRAM[0] = pitch increment")
print("  DRAM[1] = phase accumulator 1")
print("  DRAM[2] = phase accumulator 2 (modulated)")
print("  DRAM[3] = modulation depth")
print()

print("Phase 2: Waveform Lookup & Control (PC 7-12)")
print("-" * 55)
print("  PC07: B = 0                      ; clear B")
print("  PC08: A = DRAM[13]; WWF          ; WFreg = A[17:9]")
print("  PC09: XY = DRAM[14]              ; mix attenuation (pan)")
print("  PC10: B = DRAM[15]; WWF          ; B = ctrl word, WFreg update")
print("  PC11: phase += A * B             ; RP → WPHI")
print("  PC12: DRAM[15] = A + B; XY, WSP  ; update ctrl word!")
print()
print("  DRAM[13] = waveform select / envelope step")
print("  DRAM[14] = MIXL/MIXR attenuation")
print("  DRAM[15] = SLOT CONTROL WORD (I[11], ALG[10:8], M[7])")
print()

print("Phase 3: Secondary Processing (PC 13-27)")
print("-" * 55)
print("  PC13: A = DRAM[9]; XY            ; secondary mix atten")
print("  PC14: NOP (WSP)")
print("  PC15: B = A * B                  ; RP → WB")
print("  PC16: XY = DRAM[14]              ; reload mix atten")
print("  PC17: phase += DRAM[10]          ; RM 10 → WPHI")
print("  PC18: XY = A * B                 ; RP → WXY")
print("  PC19: A = DRAM[15] + A           ; read ctrl word + A")
print("  PC20: B = A * B                  ; RP → WB")
print("  PC21: A = DRAM[15] + A           ; read ctrl word + A")
print("  PC22: B = DRAM[11]; phase += B   ; RM 11 → WB, WPHI")
print("  PC23: DRAM[12] = A + B           ; RADD 12 → WM")
print("  PC24: A = B = DRAM[8]            ; RM 8 → WA, WB")
print("  PC25: A = DRAM[15] + A           ; read ctrl word + A")
print("  PC26: B = DRAM[7]; XY            ; RM 7 → WB, WXY")
print("  PC27: DRAM[7] = A + B; WSP       ; RADD 7 → WM, WSP")
print()

print("Phase 4: Output (PC 28-31)")
print("-" * 55)
print("  PC28: NOP (WSP)")
print("  PC29: WACC                       ; accumulate bus → output")
print("  PC30: NOP (WSP)")
print("  PC31: NOP (WSP)")
print()
print("  → Only 1 WACC, emitter = RSP (zero on bus)")
print("  → With all DRAM = 0: output = 0 → SILENCE")
print()

print("=== D-RAM Slot Map (16 slots) ===\n")
slot_map = {
    0:  "Pitch increment (frequency control)",
    1:  "Phase accumulator 1 (running phase)",
    2:  "Phase accumulator 2 (modulated)",
    3:  "Modulation depth",
    4:  "--- unused ---",
    5:  "--- unused ---",
    6:  "--- unused ---",
    7:  "State register (read+write, with WSP)",
    8:  "Coefficient",
    9:  "Secondary mix attenuation",
    10: "Secondary phase increment",
    11: "Secondary coefficient",
    12: "Computed output (write only)",
    13: "Waveform select / envelope step",
    14: "MIXL/MIXR output attenuation",
    15: "SLOT CONTROL WORD: X[18:12] I[11] ALG[10:8] M[7]",
}
for slot in range(16):
    print(f"  [{slot:2d}] {slot_map[slot]}")

=== SILENCE Algorithm Signal Flow ===

Phase 1: Pitch / Phase Accumulation (PC 0-6)
-------------------------------------------------------
  PC00: A = DRAM[1];  phase += A
  PC01: B = DRAM[0]
  PC02: DRAM[1] = A + B            ; phase += pitch_inc
  PC03: A = B = DRAM[2]; WSP
  PC04: DRAM[2] = A + B            ; 2nd phase accum
  PC05: B = DRAM[3]
  PC06: DRAM[2] = A + B; WSP       ; phase + mod_depth

  DRAM[0] = pitch increment
  DRAM[1] = phase accumulator 1
  DRAM[2] = phase accumulator 2 (modulated)
  DRAM[3] = modulation depth

Phase 2: Waveform Lookup & Control (PC 7-12)
-------------------------------------------------------
  PC07: B = 0                      ; clear B
  PC08: A = DRAM[13]; WWF          ; WFreg = A[17:9]
  PC09: XY = DRAM[14]              ; mix attenuation (pan)
  PC10: B = DRAM[15]; WWF          ; B = ctrl word, WFreg update
  PC11: phase += A * B             ; RP → WPHI
  PC12: DRAM[15] = A + B; XY, WSP  ; update ctrl word!

  DRAM[13] = waveform select / en

In [14]:
# D-RAM[15] is the SLOT CONTROL WORD!
# From SAM8905 programmer's guide:
# "Only address 15 of a D-RAM should hold a specific information"
print("=== D-RAM[15] = Slot Control Word (19 bits) ===\n")
print("  Bit Layout:")
print("  | 18  - 12 | 11 | 10 - 8 | 7 | 6 - 0  |")
print("  |    X     |  I |  ALG   | M |        |")
print()
print("  I    [11]   : IDLE - slot executes NOP, no WACC output")
print("  ALG  [10:8] : Algorithm select (which A-RAM block)")
print("  M    [7]    : Interrupt mask")
print("  X    [18:12]: Free (7 bits)")
print()

print("=== Self-Idling via PC12 ===\n")
print("  PC08: A = DRAM[13]; WWF      → WFreg = A[17:9]")
print("        A holds full 19-bit value of DRAM[13]")
print()
print("  PC12: DRAM[15] += A           → adds DRAM[13] to control word!")
print()
print("  The algorithm adds DRAM[13] to DRAM[15] every sample.")
print("  If DRAM[13] has upper bits set, this increments/decrements")
print("  the X field (bits 18:12). When X overflows into bit 11,")
print("  I (IDLE) is set → slot stops contributing to output.")
print()

# Demonstrate the self-idle timing
print("=== Self-Idle Timing Examples (at 44.1kHz) ===\n")
sample_rate = 44100

# If X starts at some value and we add a decrement
print("  DRAM[13] provides both:")
print("    - bits[17:9] → WF register (waveform select)")
print("    - bits[18:12] → envelope step (added to X field)")
print()

# Example: DRAM[13] = 0x7F000 means X-field decrement = 0x7F (= -1 in 7-bit)
# Starting with DRAM[15] having X=0x01
# Each sample adds 0x7F000 (which is like subtracting 0x01000)
# After 1 sample: X goes from 1 to 0... but that doesn't set IDLE.
# After 2 samples: 0 + 0x7F000 wraps... 

# Let's think in terms of when bit 11 flips
print("  When does IDLE bit flip?")
print("  Bit 11 = 0x800")
print()

# Example: start with DRAM[15] = 0x00000 (X=0, I=0, ALG=0)
# Add 0x7F000 each sample (decrement X by 1 in 7-bit 2's complement)
start = 0x00000
step = 0x7F000  # -0x01000 in 19-bit space  
mask = 0x7FFFF  # 19-bit

print(f"  Scenario: DRAM[15]=0x{start:05X}, step=0x{step:05X}")
val = start
for i in range(5):
    val = (val + step) & mask
    idle = bool(val & 0x800)
    x_field = (val >> 12) & 0x7F
    alg = (val >> 8) & 7
    print(f"    Sample {i+1}: DRAM[15]=0x{val:05X}  X=0x{x_field:02X} I={int(idle)} ALG={alg}")

print()
# Forward direction: start at 0, add positive
start2 = 0x00000
step2 = 0x01000  # increment X by 1
print(f"  Scenario: DRAM[15]=0x{start2:05X}, step=+0x{step2:05X}")
val = start2
for i in range(5):
    val = (val + step2) & mask
    idle = bool(val & 0x800)
    x_field = (val >> 12) & 0x7F
    alg = (val >> 8) & 7
    if i < 3 or idle:
        print(f"    Sample {i+1}: DRAM[15]=0x{val:05X}  X=0x{x_field:02X} I={int(idle)} ALG={alg}")

print()
print("  → IDLE bit (0x800) is BELOW the X field (bits 18:12)")
print("  → Underflow of X field does NOT propagate to bit 11!")
print("  → The firmware must set IDLE explicitly, OR...")
print("  → Use a step value that directly affects bit 11")
print()

# What about a step that directly hits bit 11?
print("  Step = 0x800 (directly toggles bit 11 each sample):")
start3 = 0x00000
step3 = 0x00800
val = start3
for i in range(4):
    val = (val + step3) & mask
    idle = bool(val & 0x800)
    print(f"    Sample {i+1}: I={int(idle)}")
print("  → Toggles IDLE every sample (not useful)")
print()
print("  Conclusion: Self-idle likely requires firmware to set")
print("  DRAM[13] to a value that, after N additions, makes")
print("  bit 11 of DRAM[15] become 1. The exact mechanism depends")
print("  on the initial DRAM[15] value set during voice_init.")

=== D-RAM[15] = Slot Control Word (19 bits) ===

  Bit Layout:
  | 18  - 12 | 11 | 10 - 8 | 7 | 6 - 0  |
  |    X     |  I |  ALG   | M |        |

  I    [11]   : IDLE - slot executes NOP, no WACC output
  ALG  [10:8] : Algorithm select (which A-RAM block)
  M    [7]    : Interrupt mask
  X    [18:12]: Free (7 bits)

=== Self-Idling via PC12 ===

  PC08: A = DRAM[13]; WWF      → WFreg = A[17:9]
        A holds full 19-bit value of DRAM[13]

  PC12: DRAM[15] += A           → adds DRAM[13] to control word!

  The algorithm adds DRAM[13] to DRAM[15] every sample.
  If DRAM[13] has upper bits set, this increments/decrements
  the X field (bits 18:12). When X overflows into bit 11,
  I (IDLE) is set → slot stops contributing to output.

=== Self-Idle Timing Examples (at 44.1kHz) ===

  DRAM[13] provides both:
    - bits[17:9] → WF register (waveform select)
    - bits[18:12] → envelope step (added to X field)

  When does IDLE bit flip?
  Bit 11 = 0x800

  Scenario: DRAM[15]=0x00000, step=

In [16]:
# Cross-reference: D-RAM config stream handlers vs algorithm slots
print("=== D-RAM Config Handlers → Algorithm Slot Mapping ===\n")
print("Handler    Address  Bytes  Likely D-RAM Targets")
print("-" * 60)
handlers = [
    ("00 Pitch",     "AB92", 4, "DRAM[0] pitch_inc, DRAM[3] mod_depth"),
    ("08 Waveform",  "ABFE", 4, "DRAM[13] wf_select"),
    ("10 Amplitude", "B030", 9, "DRAM[14] mix, DRAM[15] ctrl word"),
    ("18 Vel+DRAM",  "B222", 4, "Various (velocity-modulated)"),
    ("20 Output",    "B278", 4, "DRAM[14] MIXL/MIXR (terminates)"),
    ("28 Constant",  "B2D2", 1, "Writes 0x28 to current slot"),
    ("30 Skip/Re",   "B2CF", 1, "Re-dispatches next byte"),
]
for name, addr, nbytes, targets in handlers:
    print(f"  {name:<14} {addr}   {nbytes}      {targets}")

print()
print("=== Key Observations ===\n")
print("1. The SILENCE algorithm has all the infrastructure for a")
print("   basic subtractive voice: 2 oscillators, mixer.")
print()
print("2. Only 1 WACC at PC29 with RSP emitter → true zero output.")
print("   (Bus is floating/zero since no emitter drives it)")
print()
print("3. D-RAM[15] is the SLOT CONTROL WORD (I/ALG/M), read 4x")
print("   (PC10, 19, 21, 25) and written at PC12 (DRAM[15] += A).")
print("   When IDLE bit [11] is set, slot executes NOP → no WACC.")
print()
print("4. PC12 adds DRAM[13] to the control word every sample.")
print("   This can set the IDLE bit, self-silencing the slot.")
print()
print("5. With all DRAM = 0: phase stays 0, WF selects addr 0,")
print("   mix = 0, ctrl word unchanged → silence.")
print()
print("6. Slots 4-6 unused → available for other algorithms.")

=== D-RAM Config Handlers → Algorithm Slot Mapping ===

Handler    Address  Bytes  Likely D-RAM Targets
------------------------------------------------------------
  00 Pitch       AB92   4      DRAM[0] pitch_inc, DRAM[3] mod_depth
  08 Waveform    ABFE   4      DRAM[13] wf_select
  10 Amplitude   B030   9      DRAM[14] mix, DRAM[15] ctrl word
  18 Vel+DRAM    B222   4      Various (velocity-modulated)
  20 Output      B278   4      DRAM[14] MIXL/MIXR (terminates)
  28 Constant    B2D2   1      Writes 0x28 to current slot
  30 Skip/Re     B2CF   1      Re-dispatches next byte

=== Key Observations ===

1. The SILENCE algorithm has all the infrastructure for a
   basic subtractive voice: 2 oscillators, mixer.

2. Only 1 WACC at PC29 with RSP emitter → true zero output.
   (Bus is floating/zero since no emitter drives it)

3. D-RAM[15] is the SLOT CONTROL WORD (I/ALG/M), read 4x
   (PC10, 19, 21, 25) and written at PC12 (DRAM[15] += A).
   When IDLE bit [11] is set, slot executes NOP → 